<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/solutions/01_kmeans_gmm_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# U1 クラスタリング:k-means と混合ガウス分布(notebook)

本文:[`01-kmeans-gmm.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/unsupervised/01-kmeans-gmm.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

> **これは解答つきの版である。** まず演習の版(`unsupervised/01_kmeans_gmm.ipynb`)を自分で解き、答え合わせに使うこと。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

def make_three_blobs(n=300, seed=0):
    rng = np.random.default_rng(seed)
    mus = np.array([[0.0, 0.0], [3.5, 2.0], [-2.5, 3.0]])
    X = np.vstack([rng.normal(mu, [0.7, 0.9], (n // 3, 2)) for mu in mus])
    z = np.repeat(np.arange(3), n // 3)                     # 本当のクラスタ(学習には使わない)
    return X, z, mus

X, z_true, mus_true = make_three_blobs()
plt.figure(figsize=(4, 4)); plt.scatter(*X.T, s=10, c="gray"); plt.axis("equal"); plt.title("data (no labels)"); plt.show()

## 1. k-means:割り当てと中心の更新

本文 §2 の2つの手順を実装する。

- `assign`:全点と全中心の距離の2乗(形 `(N, K)`)をブロードキャストで作り、行ごとに最小の番号を返す
- `update_centers`:各クラスタに属する点の平均を新しい中心にする。点が1つもないクラスタは、ランダムに選んだ点を中心にする

**手計算**:本文の確認課題1(1次元の点 0, 1, 2, 10, 11, 12)を解く。

- 手計算:(ここに書く)

In [ ]:
def assign(X, centers):
    d2 = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)    # 形 (N, K)
    return d2.argmin(axis=1)

def update_centers(X, labels, K, rng):
    centers = np.zeros((K, X.shape[1]))
    for k in range(K):
        members = X[labels == k]
        centers[k] = members.mean(axis=0) if len(members) > 0 else X[rng.integers(len(X))]
    return centers

In [ ]:
def kmeans(X, K, init_centers=None, n_iters=100, seed=0):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), K, replace=False)] if init_centers is None else np.array(init_centers, dtype=float)
    J_hist = []
    for _ in range(n_iters):
        labels = assign(X, centers)
        J_hist.append(((X - centers[labels]) ** 2).sum())   # 本文の式 (1)
        new_centers = update_centers(X, labels, K, rng)
        if np.allclose(new_centers, centers):
            break
        centers = new_centers
    return labels, centers, np.array(J_hist)

x1d = np.array([[0.], [1.], [2.], [10.], [11.], [12.]])
lab, cen, J1 = kmeans(x1d, 2, init_centers=[[0.], [1.]])
check("確認課題1:中心は 1 と 11", np.sort(cen.ravel()), [1, 11])
check("確認課題1:最後の J は 4", J1[-1], 4.0)
labels, centers, J_hist = kmeans(X, 3)
check("J は単調に減る(増えない)", np.all(np.diff(J_hist) <= 1e-9), True)
print("J の変化:", J_hist.round(1))
print("推定した中心:\n", centers[np.argsort(centers[:, 0])].round(2), "\n本当の中心:\n", mus_true[np.argsort(mus_true[:, 0])])

### 初めの中心を変える

seed(初めの中心)を変えて 10 回実行し、最後の $J$ を比べる(本文 §2 の WARNING)。

In [ ]:
finals = [kmeans(X, 3, seed=s)[2][-1] for s in range(10)]
print("seed ごとの最後の J:", np.round(finals, 1))
X4, _, _ = make_three_blobs(seed=5)
finals4 = [kmeans(X4, 5, seed=s)[2][-1] for s in range(10)]
print("K = 5 にすると、seed で J が変わりやすい:", np.round(finals4, 1))

## 2. 混合ガウス分布の EM

本文 §4 を実装する。正規分布の確率密度 `gaussian_pdf` は用意してある。

- **E ステップ**:本文の式 (3) の負担率 `resp`(形 `(N, K)`)を計算する。分子を並べた `(N, K)` の配列を作り、行ごとの和で割る
- **M ステップ**:本文の式 (4) の $\pi_k$ と $\mu_k$ を計算する($\Sigma_k$ は用意してある)

**予想**:EM の対数尤度は、くり返しのたびにどう変わるか。

- 予想:(ここに書く)

In [ ]:
def gaussian_pdf(X, mu, cov):
    D = X.shape[1]
    diff = X - mu
    inv = np.linalg.inv(cov)
    norm = 1.0 / np.sqrt((2 * np.pi) ** D * np.linalg.det(cov))
    return norm * np.exp(-0.5 * np.einsum("ni,ij,nj->n", diff, inv, diff))

def e_step(X, pis, mus, covs):
    num = np.column_stack([pis[k] * gaussian_pdf(X, mus[k], covs[k]) for k in range(len(pis))])
    return num / num.sum(axis=1, keepdims=True)

def m_step(X, resp):
    N, K = resp.shape
    Nk = resp.sum(axis=0)
    pis = Nk / N
    mus = (resp.T @ X) / Nk[:, None]
    covs = []
    for k in range(K):
        diff = X - mus[k]
        covs.append((resp[:, k, None] * diff).T @ diff / Nk[k] + 1e-6 * np.eye(X.shape[1]))   # 特異にならないよう小さな値を足す
    return pis, mus, np.array(covs)

In [ ]:
def log_likelihood(X, pis, mus, covs):
    return np.log(sum(pis[k] * gaussian_pdf(X, mus[k], covs[k]) for k in range(len(pis)))).sum()

def gmm_em(X, K, n_iters=200, seed=0):
    labels, centers, _ = kmeans(X, K, seed=seed)           # k-means で初めの値を決める(本文 §5)
    pis = np.bincount(labels, minlength=K) / len(X)
    mus = centers.copy()
    covs = np.array([np.cov(X[labels == k].T) + 1e-6 * np.eye(X.shape[1]) for k in range(K)])
    ll = [log_likelihood(X, pis, mus, covs)]
    for _ in range(n_iters):
        resp = e_step(X, pis, mus, covs)
        pis, mus, covs = m_step(X, resp)
        ll.append(log_likelihood(X, pis, mus, covs))
        if ll[-1] - ll[-2] < 1e-8:
            break
    return pis, mus, covs, resp, np.array(ll)

pis, mus, covs, resp, ll = gmm_em(X, 3)
check("負担率は、各点で和が 1", resp.sum(axis=1), np.ones(len(X)))
check("対数尤度は単調に増える(減らない)", np.all(np.diff(ll) >= -1e-8), True)
check("混合の割合は、ほぼ 1/3 ずつ", np.sort(pis), np.full(3, 1 / 3), tol=0.05)
print("対数尤度:", ll.round(2))

## 3. はっきりした割り当てと、確率での割り当て

クラスタの境目にある点では、負担率が複数のクラスタに分かれる(本文 §3)。

In [ ]:
pts = np.array([[0.0, 0.0], [1.7, 1.0], [-1.2, 1.5], [3.5, 2.0]])
for p_, r in zip(pts, e_step(pts, pis, mus, covs)):
    print(f"点 {p_}:負担率 {r.round(3)}   最も近い中心(はっきりした割り当て) {assign(p_[None], mus)[0]}")
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(*X.T, c=assign(X, centers), cmap="viridis", s=10); ax[0].set_title("k-means (hard)")
ax[1].scatter(*X.T, c=resp, s=10); ax[1].set_title("GMM responsibilities as RGB (soft)")
for a in ax: a.set_aspect("equal")
plt.tight_layout(); plt.show()

## 4. 細長いクラスタ

細長く、傾いた2つのクラスタを作り、k-means と混合ガウス分布で分ける。

**予想**:どちらが本当のクラスタに近く分けるか。

- 予想:(ここに書く)

In [ ]:
rng = np.random.default_rng(3)
C = np.array([[3.0, 2.8], [2.8, 3.0]])                     # 斜めに細長い共分散
Xe = np.vstack([rng.multivariate_normal([0, 0], C, 200), rng.multivariate_normal([0, 2.5], C, 200)])
ze = np.repeat([0, 1], 200)
lab_km = kmeans(Xe, 2, seed=0)[0]
lab_gm = gmm_em(Xe, 2, seed=0)[3].argmax(axis=1)
agree = lambda a, b: max(np.mean(a == b), np.mean(a != b))  # クラスタの番号の入れ替えは気にしない
print(f"本当のクラスタとの一致:k-means {agree(lab_km, ze):.3f}   混合ガウス分布 {agree(lab_gm, ze):.3f}")
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, l, t in zip(ax, [ze, lab_km, lab_gm], ["true", "k-means", "GMM"]):
    a.scatter(*Xe.T, c=l, cmap="coolwarm", s=8); a.set_title(t); a.set_aspect("equal")
plt.tight_layout(); plt.show()

## 5. クラスタの数を選ぶ

$K = 1, \dots, 6$ で、k-means の $J$ と、混合ガウス分布の BIC(= $-2 \times$ 対数尤度 $+$ パラメータの数 $\times \log N$。小さいほど良い)を比べる(本文 §6)。

In [ ]:
Ks = range(1, 7)
Js = [min(kmeans(X, K, seed=s)[2][-1] for s in range(5)) for K in Ks]
bics = []
for K in Ks:
    pis_k, mus_k, covs_k, _, ll_k = gmm_em(X, K)
    n_params = (K - 1) + K * 2 + K * 3                     # 混合の割合、平均、共分散(2次元なので3つ)
    bics.append(-2 * ll_k[-1] + n_params * np.log(len(X)))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(Ks, Js, "o-"); ax[0].set_xlabel("K"); ax[0].set_ylabel("J (k-means)")
ax[1].plot(Ks, bics, "o-"); ax[1].set_xlabel("K"); ax[1].set_ylabel("BIC (GMM)")
plt.tight_layout(); plt.show()
print("BIC が最小の K:", list(Ks)[int(np.argmin(bics))])

## 6. 条件を変える:確率密度が小さい点(異常の候補)

混合ガウス分布の確率密度 $p(x)$(本文の式 (2))が小さい点は、どのクラスタからも外れている(本文 §7)。学習に使ったデータに、外れた点を5つ足して、 $p(x)$ の小さい順に並べる。

1. 外れた点は、 $p(x)$ の小さいほうに並ぶか
2. 外れた点を学習に含めた場合と含めない場合で、結果は変わるか

In [ ]:
outliers = np.array([[6.0, -2.0], [-5.0, -2.0], [0.5, 6.0], [7.0, 5.0], [-4.5, 7.0]])
X_out = np.vstack([X, outliers])
density = lambda Z, pis, mus, covs: sum(pis[k] * gaussian_pdf(Z, mus[k], covs[k]) for k in range(len(pis)))
for name, X_fit in [("外れた点を含めずに学習", X), ("外れた点も含めて学習", X_out)]:
    pis_o, mus_o, covs_o, _, _ = gmm_em(X_fit, 3)
    order = np.argsort(density(X_out, pis_o, mus_o, covs_o))
    print(f"{name}:p(x) が小さい5点の番号 {sorted(order[:5].tolist())}(外れた点の番号は {list(range(len(X), len(X_out)))})")

## 7. 研究で使うときは

`cpai_lab` には、クラスタリングの関数は入っていない(本文 §8)。scikit-learn の `KMeans` と `GaussianMixture` は、この notebook と同じことをする。

| この notebook | scikit-learn |
|---|---|
| seed を変えて何回か実行し、 $J$ が最小のものを選ぶ | `KMeans(n_init=10)` |
| `kmeans` の $J$ | `KMeans(...).inertia_` |
| `e_step` の負担率 | `GaussianMixture(...).predict_proba(X)` |
| `density` の対数 | `GaussianMixture(...).score_samples(X)` |
| BIC | `GaussianMixture(...).bic(X)` |

## 8. 説明(自分の言葉で書く)

1. k-means の2つの手順は、それぞれ何をしているか。 $J$ が増えないのはなぜか
   - (ここに書く)
2. 負担率とは何か。k-means の割り当てとどう違うか
   - (ここに書く)
3. 細長いクラスタで、k-means と混合ガウス分布の結果が違ったのはなぜか
   - (ここに書く)